<a href="https://colab.research.google.com/github/MightyNine/The-Mighty-Nein/blob/main/GAN%20Model%20CICIoT2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Mount Google Drive**

In [ ]:
# Import Colab's Google Drive helper so we can read the dataset files stored in Drive
from google.colab import drive

# Mount Drive at /content/drive (Colab shows a permission pop-up the first time you run this)
drive.mount('/content/drive')

**2.Imports, Seed and Device**

In [ ]:
# Standard library tools
import os          # file paths and folder creation
import time        # timing for the efficiency profile
import copy        # copying the model for the CPU latency test
import random      # Python's built-in random generator (seeded below)

# Core data libraries
import numpy as np     # numerical arrays
import pandas as pd    # loading and cleaning the CSV data

# PyTorch: the deep learning framework used for the GAN
import torch                          # tensors and GPU support
import torch.nn as nn                 # neural network layers and loss functions
import torch.optim as optim           # optimisers (Adam)
from torch.utils.data import DataLoader   # feeds data to the model in batches

# Plotting
import matplotlib.pyplot as plt                # static charts
import matplotlib.animation as animation       # live-demo animation engine
from matplotlib.lines import Line2D            # custom legend entries for the demo
from IPython.display import HTML               # renders the animation inline in Colab

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,            # overall accuracy
    balanced_accuracy_score,   # accuracy averaged over both classes (fair under imbalance)
    precision_score,           # precision for the anomaly class
    recall_score,              # recall for the anomaly class
    f1_score,                  # F1 for the anomaly class
    classification_report,     # per-class precision/recall/F1 table
    confusion_matrix,          # TP/FP/TN/FN counts
    ConfusionMatrixDisplay,    # plots the confusion matrix
    roc_auc_score,             # area under the ROC curve
    roc_curve,                 # points for plotting the ROC curve
)

# Fixed seed so results are reproducible across reruns
SEED = 42
random.seed(SEED)                       # seed Python's random module
np.random.seed(SEED)                    # seed NumPy
torch.manual_seed(SEED)                 # seed PyTorch on CPU
torch.cuda.manual_seed_all(SEED)        # seed PyTorch on every GPU (safe if no GPU)

# Use the GPU if Colab gave us one, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")                                         # show which device is in use
if device.type == "cuda":                                          # only runs if a GPU is available
    print(f"GPU: {torch.cuda.get_device_name(0)}")                 # show the GPU model name

**3. Configuration (Paths and Hyperparamaters)**

In [ ]:
DRIVE_FOLDER = "/content/drive/MyDrive/"

TRAIN_CSV = os.path.join(DRIVE_FOLDER, "train.csv")        # full path to the training file
TEST_CSV  = os.path.join(DRIVE_FOLDER, "test.csv")         # full path to the testing file
CHECKPOINT_DIR = os.path.join(DRIVE_FOLDER, "gan_checkpoints")   # where trained weights get saved
os.makedirs(CHECKPOINT_DIR, exist_ok=True)                 # create the checkpoint folder if missing

# Stop early with a clear message if the files aren't where we expect
assert os.path.exists(TRAIN_CSV), f"train.csv not found at {TRAIN_CSV} - check DRIVE_FOLDER"
assert os.path.exists(TEST_CSV),  f"test.csv not found at {TEST_CSV} - check DRIVE_FOLDER"

# ---- Data settings ----
CHUNK_SIZE = 500_000          # rows read at a time from the huge train.csv (keeps RAM usage low)
MAX_TRAIN_SAMPLES = 100_000   # cap on benign samples used for GAN training and validation
VAL_FRACTION = 0.15           # share of benign samples held out for validation/threshold calibration
DROP_OUT_OF_SCOPE = True      # drop attack types outside our 4 target categories (matches the LSTM notebook)

# ---- GAN settings ----
Z_DIM = 100          # length of the random noise vector fed into the Generator
BATCH_SIZE = 256     # samples per training step
EPOCHS = 30          # passes over the training data
LR = 0.0002          # learning rate for both networks (standard DCGAN value)
REAL_LABEL = 0.9     # "soft" label for real data; stops the Discriminator becoming overconfident

# ---- Detection settings ----
THRESHOLD_PERCENTILE = 95    # anomaly threshold = this percentile of benign validation scores

4. **Load the Dataset from Drive**

In [ ]:
# Load the test set fully (about 340 MB, fits comfortably in memory)
test_df = pd.read_csv(TEST_CSV)

# Find the label column name (it differs slightly between dataset copies)
possible_label_names = ["label", "Label", " Label"]                        # common variants
LABEL_COL = next((c for c in possible_label_names if c in test_df.columns), None)   # first match
assert LABEL_COL is not None, f"No label column found. Columns: {test_df.columns.tolist()}"
print(f"Label column: '{LABEL_COL}'")

# Check whether train.csv has a header row (some copies of the dataset don't)
train_header = pd.read_csv(TRAIN_CSV, nrows=0).columns          # read column names only
if LABEL_COL in train_header:
    read_kwargs = {}                                             # file has headers: read normally
else:
    read_kwargs = dict(header=None, names=test_df.columns)       # no headers: reuse the test file's names

# train.csv is about 1.5 GB, so we read it in chunks and keep ONLY benign rows
# (the GAN only ever trains on normal traffic, so attack rows aren't needed here)
benign_chunks = []     # will hold the benign rows from each chunk
total_train_rows = 0   # running count of rows scanned, for the summary below

for chunk in pd.read_csv(TRAIN_CSV, chunksize=CHUNK_SIZE, **read_kwargs):
    total_train_rows += len(chunk)                                            # count rows scanned
    is_benign = chunk[LABEL_COL].astype(str).str.lower().str.contains("benign")   # True for benign rows
    benign_chunks.append(chunk[is_benign])                                    # keep only the benign rows

benign_df = pd.concat(benign_chunks, ignore_index=True)    # combine all benign rows into one DataFrame
del benign_chunks                                          # free memory

print(f"Scanned {total_train_rows:,} training rows -> kept {len(benign_df):,} benign rows")
print(f"Test set shape: {test_df.shape}")
print("\nTest label distribution (raw labels):")
print(test_df[LABEL_COL].value_counts())

**5. Map Labels to the 5 Project Categories**

In [ ]:
# Map CICIoT2023's many granular attack names into the categories from the project proposal.
# This is the same mapping used in the LSTM notebook, so results are directly comparable.

def map_to_category(raw_label):
    """Return Benign, DoS, DDoS, BruteForce, Botnet or Other for a raw label string."""
    label_lower = str(raw_label).lower()                               # lowercase for matching
    if "benign" in label_lower:                                        # normal traffic
        return "Benign"
    elif "ddos" in label_lower:                                        # must be checked BEFORE 'dos'
        return "DDoS"
    elif "dos" in label_lower:                                         # single-source denial of service
        return "DoS"
    elif "brute" in label_lower or "dictionary" in label_lower:        # credential guessing attacks
        return "BruteForce"
    elif "mirai" in label_lower or "botnet" in label_lower:            # botnet-driven traffic
        return "Botnet"
    else:
        return "Other"                                                 # outside the project scope

test_df["category"] = test_df[LABEL_COL].apply(map_to_category)        # add the mapped category column

# Optionally drop attack types the proposal doesn't cover (Recon, Spoofing, Web attacks, etc.)
if DROP_OUT_OF_SCOPE:
    before = len(test_df)                                              # row count before dropping
    test_df = test_df[test_df["category"] != "Other"].reset_index(drop=True)   # keep in-scope rows only
    print(f"Dropped {before - len(test_df):,} out-of-scope test rows")

print("\nTest category distribution:")
print(test_df["category"].value_counts())

**6. Clean Features, Split Benign Data, Fit Scaling**

In [ ]:
# Pick the numeric feature columns (everything except the label and category columns)
non_feature_cols = [LABEL_COL, "category"]
feature_cols = [
    c for c in benign_df.columns
    if c not in non_feature_cols and pd.api.types.is_numeric_dtype(benign_df[c])
]
n_features = len(feature_cols)                                        # how many features feed the model
assert 0 < n_features <= 256, f"{n_features} features won't fit a 16x16 grid (max 256)"
print(f"Using {n_features} numeric features")

# Replace infinite values with NaN so they can be filled uniformly below
benign_X = benign_df[feature_cols].replace([np.inf, -np.inf], np.nan)
del benign_df                                                         # free memory

# Cap the number of benign samples to keep training fast
if len(benign_X) > MAX_TRAIN_SAMPLES:
    benign_X = benign_X.sample(n=MAX_TRAIN_SAMPLES, random_state=SEED)

# Split benign data into a training portion and a validation portion
rng = np.random.default_rng(SEED)                                     # seeded random generator
perm = rng.permutation(len(benign_X))                                 # shuffled row positions
n_val = int(len(benign_X) * VAL_FRACTION)                             # validation set size
val_df = benign_X.iloc[perm[:n_val]]                                  # held-out benign validation rows
train_df = benign_X.iloc[perm[n_val:]]                                # benign training rows

# Fill missing values using medians from the TRAINING portion only (no leakage from val/test)
medians = train_df.median().fillna(0)                                 # fillna(0) covers all-NaN columns
train_df = train_df.fillna(medians)
val_df = val_df.fillna(medians)

# Network features are heavily skewed (a few huge values), so we compress them with a signed log
# before scaling. Without this, min-max scaling would squash most values into a tiny range.
def signed_log(values):
    """Apply sign(x) * log(1 + |x|) element-wise: compresses big values, preserves sign."""
    return np.sign(values) * np.log1p(np.abs(values))

# Learn the scaling range from benign TRAINING data only
log_train = signed_log(train_df.to_numpy(dtype=np.float64))          # log-compressed training features
data_min = log_train.min(axis=0)                                      # per-feature minimum
data_max = log_train.max(axis=0)                                      # per-feature maximum
data_range = np.where((data_max - data_min) == 0, 1.0, data_max - data_min)   # avoid divide-by-zero

print(f"Benign train: {len(train_df):,} | Benign validation: {len(val_df):,}")

**7. Coverting Everything to 16x16 Tensors**

In [ ]:
def to_gan_tensor(df):
    """Turn a DataFrame of features into a (N, 1, 16, 16) tensor scaled to [-1, 1]."""
    values = signed_log(df.to_numpy(dtype=np.float64))                # compress heavy tails
    scaled = 2 * (values - data_min) / data_range - 1                 # scale to [-1, 1] using TRAINING range
    scaled = np.clip(scaled, -3, 3).astype(np.float32)                # safety clip for extreme test outliers
    padded = np.zeros((len(scaled), 256), dtype=np.float32)           # blank 256-slot rows (16 x 16)
    padded[:, :n_features] = scaled                                   # features first, zero-padding after
    return torch.from_numpy(padded.reshape(-1, 1, 16, 16))            # reshape into 1-channel 16x16 images

train_tensor = to_gan_tensor(train_df)       # benign training images
val_tensor = to_gan_tensor(val_df)           # benign validation images

# Prepare the test set using the SAME columns, medians and scaling as the training data
test_features = (
    test_df.reindex(columns=feature_cols)            # same columns, same order as training
    .replace([np.inf, -np.inf], np.nan)              # remove infinities
    .fillna(medians)                                 # fill gaps with TRAINING medians
    .fillna(0)                                       # last-resort fill for any remaining gaps
)
test_tensor = to_gan_tensor(test_features)           # test images (benign + attacks)

# Ground truth for the test set
y_true = (test_df["category"] != "Benign").astype(int).to_numpy()    # 0 = Benign, 1 = Anomaly
test_cat = test_df["category"].to_numpy()                            # category name per test row

# Batch loader for training (drop_last avoids a tiny final batch that upsets BatchNorm)
train_loader = DataLoader(train_tensor, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Train tensor: {tuple(train_tensor.shape)}")
print(f"Val tensor:   {tuple(val_tensor.shape)}")
print(f"Test tensor:  {tuple(test_tensor.shape)}  (benign={np.sum(y_true == 0):,}, anomaly={np.sum(y_true == 1):,})")

**8. Define the Generator and Discriminator**

In [ ]:
class Generator(nn.Module):
    """Turns a random noise vector into a fake 16x16 'network traffic image'."""
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 256, 4, 1, 0),   # noise (1x1) -> 4x4 feature map
            nn.BatchNorm2d(256),                       # stabilises training
            nn.ReLU(True),                             # non-linearity
            nn.ConvTranspose2d(256, 128, 4, 2, 1),     # 4x4 -> 8x8
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),       # 8x8 -> 16x16, single channel
            nn.Tanh(),                                 # output in [-1, 1], matching our scaled data
        )
    def forward(self, z):
        return self.net(z)                             # run noise through the network

class Discriminator(nn.Module):
    """Scores a 16x16 image: near 1 = looks like normal traffic, near 0 = looks fake/abnormal."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 128, 4, 2, 1),                # 16x16 -> 8x8
            nn.LeakyReLU(0.2, inplace=True),           # keeps small gradients for negative values
            nn.Conv2d(128, 256, 4, 2, 1),              # 8x8 -> 4x4
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, 4, 1, 0),                # 4x4 -> single score
            nn.Sigmoid(),                              # squash to [0, 1]
        )
    def forward(self, x):
        return self.net(x)                             # run the image through the network

def weights_init(m):
    """Standard DCGAN weight initialisation (small random values)."""
    name = m.__class__.__name__                                       # layer type as a string
    if "Conv" in name:                                                # Conv and ConvTranspose layers
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in name:                                         # BatchNorm layers
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

generator = Generator(Z_DIM).to(device)           # build the Generator on the chosen device
discriminator = Discriminator().to(device)        # build the Discriminator on the chosen device
generator.apply(weights_init)                     # initialise Generator weights
discriminator.apply(weights_init)                 # initialise Discriminator weights

criterion = nn.BCELoss()                                                      # binary cross-entropy loss
optimizer_G = optim.Adam(generator.parameters(), lr=LR, betas=(0.5, 0.999))   # Generator optimiser
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR, betas=(0.5, 0.999))   # Discriminator optimiser

def discriminator_scores(tensor, batch_size=4096):
    """Return the Discriminator's score (0-1) for every image in a tensor."""
    discriminator.eval()                                      # evaluation mode (BatchNorm uses running stats)
    out = []                                                  # collects scores batch by batch
    with torch.no_grad():                                     # no gradients needed for scoring
        for i in range(0, len(tensor), batch_size):
            batch = tensor[i:i + batch_size].to(device)       # move one batch to the device
            out.append(discriminator(batch).view(-1).cpu().numpy())   # score it and bring back to CPU
    return np.concatenate(out)                                # join all batches into one array

print(f"Generator parameters:     {sum(p.numel() for p in generator.parameters()):,}")
print(f"Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}")

**9. Training the GAN Model**

In [ ]:
# Track training statistics per epoch so we can plot them afterwards
history = {"loss_D": [], "loss_G": [], "D_real": [], "D_fake": [], "val_D": []}

for epoch in range(EPOCHS):
    generator.train()                  # training mode for the Generator
    discriminator.train()              # training mode for the Discriminator
    epoch_start = time.time()          # start the epoch timer
    sums = {"loss_D": 0.0, "loss_G": 0.0, "D_real": 0.0, "D_fake": 0.0}   # running totals this epoch
    n_batches = 0                      # number of batches processed this epoch

    for real in train_loader:
        real = real.to(device)                         # move the real batch to the device
        b = real.size(0)                               # batch size

        # ---------- Train the Discriminator ----------
        optimizer_D.zero_grad()                                              # clear old gradients
        real_out = discriminator(real).view(-1)                              # scores for real benign data
        loss_real = criterion(real_out, torch.full_like(real_out, REAL_LABEL))   # real should score ~0.9
        noise = torch.randn(b, Z_DIM, 1, 1, device=device)                   # random noise vectors
        fake = generator(noise)                                              # Generator's fake images
        fake_out = discriminator(fake.detach()).view(-1)                     # scores for fakes (detach: don't train G here)
        loss_fake = criterion(fake_out, torch.zeros_like(fake_out))          # fakes should score ~0
        loss_D = loss_real + loss_fake                                       # total Discriminator loss
        loss_D.backward()                                                    # compute gradients
        optimizer_D.step()                                                   # update Discriminator weights

        # ---------- Train the Generator ----------
        optimizer_G.zero_grad()                                              # clear old gradients
        g_out = discriminator(fake).view(-1)                                 # Discriminator's verdict on the fakes
        loss_G = criterion(g_out, torch.ones_like(g_out))                    # Generator wants fakes scored as real
        loss_G.backward()                                                    # compute gradients
        optimizer_G.step()                                                   # update Generator weights

        # Accumulate this batch's statistics
        sums["loss_D"] += loss_D.item()
        sums["loss_G"] += loss_G.item()
        sums["D_real"] += real_out.mean().item()
        sums["D_fake"] += fake_out.mean().item()
        n_batches += 1

    # Average the statistics over the epoch and record them
    for key in sums:
        history[key].append(sums[key] / n_batches)

    # Mean Discriminator score on held-out benign data (should stay high if it recognises normal traffic)
    history["val_D"].append(float(discriminator_scores(val_tensor).mean()))

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Loss D: {history['loss_D'][-1]:.4f} | "
          f"Loss G: {history['loss_G'][-1]:.4f} | D(real): {history['D_real'][-1]:.3f} | "
          f"D(fake): {history['D_fake'][-1]:.3f} | Val D: {history['val_D'][-1]:.3f} | "
          f"{time.time() - epoch_start:.1f}s")

    # Save a checkpoint to Drive after every epoch so a Colab disconnect doesn't lose progress
    torch.save({
        "epoch": epoch + 1,
        "generator": generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "feature_cols": feature_cols,
        "data_min": data_min,
        "data_max": data_max,
        "medians": medians,
    }, os.path.join(CHECKPOINT_DIR, "gan_latest.pt"))

print("\nTraining complete. Checkpoint saved to Drive.")

**10. Training the Curves**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))   # two side-by-side panels
epochs_axis = np.arange(1, EPOCHS + 1)                    # x-axis: epoch numbers

# Left panel: Generator and Discriminator losses
ax1.plot(epochs_axis, history["loss_D"], label="Discriminator loss", color="steelblue")
ax1.plot(epochs_axis, history["loss_G"], label="Generator loss", color="firebrick")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("GAN Training Losses")
ax1.legend()
ax1.grid(alpha=0.3)

# Right panel: what the Discriminator thinks of real, fake and held-out benign data
ax2.plot(epochs_axis, history["D_real"], label="D(real train)", color="seagreen")
ax2.plot(epochs_axis, history["D_fake"], label="D(fake)", color="firebrick")
ax2.plot(epochs_axis, history["val_D"], label="D(benign validation)", color="darkblue", linestyle="--")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Mean Discriminator Score")
ax2.set_title("Discriminator Scores Over Training")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**11. Generator Quality Check (Real vs Generated Feature Profiles)**

In [ ]:
# Does the Generator produce traffic with a similar feature profile to real benign traffic?
generator.eval()                                                     # evaluation mode
with torch.no_grad():                                                # no gradients needed
    noise = torch.randn(2000, Z_DIM, 1, 1, device=device)            # 2000 random noise vectors
    fake_feats = generator(noise).cpu().view(-1, 256)[:, :n_features].numpy()   # generated features (drop padding)

real_feats = val_tensor[:2000].view(-1, 256)[:, :n_features].numpy()   # 2000 real benign validation samples

real_mean = real_feats.mean(axis=0)       # average of each feature across real samples
fake_mean = fake_feats.mean(axis=0)       # average of each feature across generated samples
corr = np.corrcoef(real_mean, fake_mean)[0, 1]   # how closely the two profiles agree (1 = identical shape)

plt.figure(figsize=(11, 4.5))
plt.plot(real_mean, label="Real benign (validation)", color="seagreen", marker="o", markersize=3)
plt.plot(fake_mean, label="Generated (fake)", color="firebrick", marker="o", markersize=3)
plt.xlabel("Feature index")
plt.ylabel("Mean scaled value")
plt.title(f"Feature Profile: Real vs Generated Traffic (correlation = {corr:.3f})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**12. Score the Test Set**

In [ ]:
# The Discriminator outputs "how normal does this look" (high = normal).
# We flip it into an ANOMALY score (high = anomalous) so it matches the LSTM notebook's convention.

val_anom = 1.0 - discriminator_scores(val_tensor)       # anomaly scores for benign validation data
test_anom = 1.0 - discriminator_scores(test_tensor)     # anomaly scores for the whole test set

print(f"Validation (benign) anomaly score: mean={val_anom.mean():.4f}, std={val_anom.std():.4f}")
print(f"Test benign anomaly score:         mean={test_anom[y_true == 0].mean():.4f}")
print(f"Test attack anomaly score:         mean={test_anom[y_true == 1].mean():.4f}")

**13. Calibrating the Threshold and Evaluate**

In [ ]:
# Threshold calibration: use the 95th percentile of BENIGN validation anomaly scores.
# This is the same method as the LSTM notebook, so both models are judged on equal footing.
threshold = np.percentile(val_anom, THRESHOLD_PERCENTILE)
y_pred = (test_anom > threshold).astype(int)             # 1 = flagged as anomaly
print(f"Calibrated threshold ({THRESHOLD_PERCENTILE}th percentile of benign validation): {threshold:.4f}")

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=["Benign", "Anomaly"], cmap="Blues"
)
plt.title("GAN Confusion Matrix (Calibrated Threshold)")
plt.tight_layout()
plt.show()

# Per-class precision, recall and F1
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["Benign", "Anomaly"], zero_division=0))

# ROC-AUC uses the continuous score, so it doesn't depend on the chosen threshold
auc = roc_auc_score(y_true, test_anom)
print(f"ROC-AUC: {auc:.4f}")

# ROC curve plot
fpr, tpr, _ = roc_curve(y_true, test_anom)
plt.figure(figsize=(6, 5.5))
plt.plot(fpr, tpr, color="steelblue", linewidth=2, label=f"GAN (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("GAN ROC Curve (Anomaly vs Benign)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

# Compare our calibrated threshold against the naive fixed 0.5 rule used in earlier GAN notebooks
y_pred_fixed = (test_anom > 0.5).astype(int)             # discriminator score below 0.5 = anomaly

def summarise(preds):
    """Collect headline metrics for a set of predictions."""
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()          # unpack the confusion matrix
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Balanced Acc": balanced_accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds, zero_division=0),
        "Recall": recall_score(y_true, preds, zero_division=0),
        "F1": f1_score(y_true, preds, zero_division=0),
        "False Alarm Rate": fp / (fp + tn) if (fp + tn) else 0.0,
    }

comparison = pd.DataFrame({
    f"Calibrated ({threshold:.3f})": summarise(y_pred),
    "Fixed 0.5": summarise(y_pred_fixed),
}).round(4)
print("\nThreshold comparison:")
print(comparison)

**14. Per-Attack-Type Detection and AUC**

In [ ]:
categories = ["Benign", "DoS", "DDoS", "BruteForce", "Botnet"]   # the 5 project categories
rates = {}                                                       # correct-classification rate per category
cat_auc = {}                                                     # one-vs-benign AUC per attack type

for cat in categories:
    mask = test_cat == cat                                       # rows belonging to this category
    if mask.sum() == 0:                                          # skip categories absent from the test set
        continue
    flagged = (test_anom[mask] > threshold)                      # which of these rows were flagged anomalous
    rates[cat] = (~flagged).mean() * 100 if cat == "Benign" else flagged.mean() * 100   # benign: kept normal; attacks: caught
    print(f"{cat:11s} | n={mask.sum():7,d} | correct rate: {rates[cat]:6.2f}%")

    if cat != "Benign":                                          # AUC for this attack vs benign traffic
        subset = mask | (test_cat == "Benign")                   # this attack + all benign rows
        cat_auc[cat] = roc_auc_score((test_cat[subset] == cat).astype(int), test_anom[subset])

# Bar chart of correct-classification rate per category
plt.figure(figsize=(8, 5))
bars = plt.bar(rates.keys(), rates.values(),
               color=["seagreen" if k == "Benign" else "firebrick" for k in rates])
plt.ylabel("Correct Classification Rate (%)")
plt.title("GAN Detection Performance by Traffic Category")
plt.ylim(0, 105)
plt.grid(axis="y", alpha=0.3)
for bar, value in zip(bars, rates.values()):                     # label each bar with its value
    plt.text(bar.get_x() + bar.get_width() / 2, value + 1, f"{value:.1f}%", ha="center")
plt.show()

print("\nOne-vs-benign ROC-AUC by attack type:")
print(pd.Series(cat_auc).sort_values(ascending=False).round(4))

**15. Anomaly Score Distribution by Category**

In [ ]:
plt.figure(figsize=(10, 5.5))
colors = {"Benign": "seagreen", "DoS": "orange", "DDoS": "firebrick",
          "BruteForce": "purple", "Botnet": "darkblue"}          # one colour per category

for cat in categories:
    mask = test_cat == cat
    if mask.sum() == 0:
        continue
    plt.hist(test_anom[mask], bins=50, alpha=0.5, density=True,
             label=f"{cat} (n={mask.sum():,})", color=colors[cat])

plt.axvline(threshold, color="black", linestyle="--", linewidth=2, label="Calibrated threshold")
plt.xlabel("Anomaly Score (1 - Discriminator score)")
plt.ylabel("Density")
plt.title("GAN Anomaly Score Distribution by Traffic Category")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.show()

**16. Threshold Sensitivity Analysis**

In [ ]:
# Test a range of thresholds to justify the 95th percentile choice.
# We track balanced accuracy as well as F1, because F1 alone rewards "flag everything"
# when the test set is overwhelmingly attack traffic.
percentiles = np.arange(80, 99.6, 0.5)      # percentiles of benign validation scores to try
rows = []
for pct in percentiles:
    t = np.percentile(val_anom, pct)                          # candidate threshold
    preds = (test_anom > t).astype(int)                       # predictions at this threshold
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()  # confusion matrix counts
    rows.append({
        "percentile": pct,
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, preds),
        "false_alarm_rate": fp / (fp + tn) if (fp + tn) else 0.0,
    })
sweep = pd.DataFrame(rows)

best = sweep.loc[sweep["balanced_acc"].idxmax()]            # threshold with the best balanced accuracy
print(f"Best balanced accuracy at the {best['percentile']:.1f}th percentile: {best['balanced_acc']:.3f}")
print(f"Chosen threshold ({THRESHOLD_PERCENTILE}th percentile): "
      f"{sweep.loc[sweep['percentile'] == THRESHOLD_PERCENTILE, 'balanced_acc'].values[0]:.3f}")

plt.figure(figsize=(10, 5.5))
plt.plot(sweep["percentile"], sweep["precision"], label="Precision", color="steelblue")
plt.plot(sweep["percentile"], sweep["recall"], label="Recall", color="firebrick")
plt.plot(sweep["percentile"], sweep["f1"], label="F1", color="seagreen")
plt.plot(sweep["percentile"], sweep["balanced_acc"], label="Balanced accuracy", color="purple")
plt.plot(sweep["percentile"], sweep["false_alarm_rate"], label="False alarm rate", color="gray", linestyle=":")
plt.axvline(THRESHOLD_PERCENTILE, color="black", linestyle="--", alpha=0.7, label="Chosen threshold")
plt.xlabel("Threshold Percentile (of benign validation scores)")
plt.ylabel("Score")
plt.title("GAN Threshold Sensitivity")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**17. Model Efficiency Profile (IoT Deployment Feasibility)**

In [ ]:
# Only the Discriminator is needed at detection time (the Generator is just for training),
# so this profiles the Discriminator as the deployable detector.

n_params_D = sum(p.numel() for p in discriminator.parameters())          # parameter count

tmp_path = "/content/tmp_discriminator.pt"                               # temporary file to measure size
torch.save(discriminator.state_dict(), tmp_path)                         # save weights
size_mb = os.path.getsize(tmp_path) / (1024 ** 2)                        # file size in MB
os.remove(tmp_path)                                                      # clean up

def single_latency_ms(model, dev, n_trials=200):
    """Average time (ms) to score ONE sample."""
    model.eval()
    x = test_tensor[:1].to(dev)                                          # one sample
    with torch.no_grad():
        for _ in range(20):                                              # warm-up runs, not timed
            model(x)
        if dev.type == "cuda":
            torch.cuda.synchronize()                                     # wait for the GPU to finish
        start = time.perf_counter()
        for _ in range(n_trials):
            model(x)
        if dev.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_trials * 1000

cpu_model = copy.deepcopy(discriminator).cpu()                           # CPU copy: closer to IoT gateway hardware
latency_cpu = single_latency_ms(cpu_model, torch.device("cpu"))          # single-sample latency on CPU
latency_dev = single_latency_ms(discriminator, device)                   # single-sample latency on current device

batch = test_tensor[:4096].to(device)                                    # a batch for throughput measurement
discriminator.eval()
with torch.no_grad():
    discriminator(batch)                                                 # warm-up
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    discriminator(batch)
    if device.type == "cuda":
        torch.cuda.synchronize()
    throughput = len(batch) / (time.perf_counter() - start)              # samples scored per second

print("=" * 48)
print("GAN DISCRIMINATOR EFFICIENCY PROFILE")
print("=" * 48)
print(f"Parameters:               {n_params_D:,}")
print(f"Model size on disk:       {size_mb:.2f} MB")
print(f"Latency per sample (CPU): {latency_cpu:.2f} ms")
print(f"Latency per sample ({device.type.upper()}): {latency_dev:.2f} ms")
print(f"Batch throughput:         {throughput:,.0f} samples/sec")
print("=" * 48)

**18. Live Tracking Simulation (Two-Panel Streaming Demo)**

In [ ]:
# Replays test samples one at a time to simulate a live monitoring feed.
# TOP PANEL: anomaly score per sample, dots coloured by what the traffic ACTUALLY is.
# BOTTOM PANEL: running count of what the GAN has flagged so far, plus running stats.
# NOTE: this is a dataset REPLAY, not a live packet capture.

plt.rcParams["animation.embed_limit"] = 60     # raise the size limit so Colab doesn't truncate the animation

N_DEMO_STEPS = 300         # samples to replay
BENIGN_FRACTION = 0.35     # share of the demo that is benign (the real test set is almost all attacks)

# Build a curated, shuffled stream. SAY THIS IN THE PRESENTATION: the demo mix is curated,
# the evaluation metrics above are computed on the full, un-curated test set.
raw_names = test_df[LABEL_COL].to_numpy()                      # granular attack names for display
rng = np.random.default_rng(SEED)                              # seeded so the demo is identical every run
benign_idx = np.where(y_true == 0)[0]                          # benign row positions
attack_idx = np.where(y_true == 1)[0]                          # attack row positions
n_benign = min(int(N_DEMO_STEPS * BENIGN_FRACTION), len(benign_idx))
n_attack = min(N_DEMO_STEPS - n_benign, len(attack_idx))
demo_idx = np.concatenate([rng.choice(benign_idx, n_benign, replace=False),
                           rng.choice(attack_idx, n_attack, replace=False)])
rng.shuffle(demo_idx)                                          # interleave benign and attack traffic
N = len(demo_idx)                                              # final stream length

demo_scores = test_anom[demo_idx]                              # anomaly score per demo sample
demo_true = y_true[demo_idx]                                   # ground truth per demo sample
demo_names = raw_names[demo_idx]                               # attack name per demo sample
demo_preds = (demo_scores > threshold).astype(int)             # GAN decision per demo sample
point_colors = ["seagreen" if t == 0 else "firebrick" for t in demo_true]   # dot colour = TRUE class

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8))          # two stacked panels

# ---- Panel 1: live anomaly score stream ----
line, = ax1.plot([], [], color="lightgray", lw=1)              # faint connecting line
dots = ax1.scatter([], [], s=28, zorder=3)                     # dots coloured by true class
ax1.axhline(threshold, color="red", linestyle="--", lw=1.5)    # detection threshold
ax1.axhspan(threshold, 1.0, color="red", alpha=0.06)           # shaded "flagged as anomaly" zone
ax1.set_xlim(0, N)
ax1.set_ylim(-0.02, 1.05)
ax1.set_xlabel("Samples Seen")
ax1.set_ylabel("Anomaly Score")
ax1.set_title("Live GAN Anomaly Score Stream (CICIoT2023 Test Replay)")
ax1.grid(alpha=0.3)
ax1.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="seagreen", markersize=9, label="Actual: Benign"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="firebrick", markersize=9, label="Actual: Attack"),
    Line2D([0], [0], color="red", linestyle="--", label=f"Threshold ({threshold:.3f})"),
], loc="center right", fontsize=9)
status = ax1.text(0.01, 0.96, "", transform=ax1.transAxes, va="top", fontsize=10, fontweight="bold")

# ---- Panel 2: running flagged counts ----
bars = ax2.bar(["Flagged Normal", "Flagged Anomaly"], [0, 0], color=["seagreen", "firebrick"])
ax2.set_ylim(0, N)
ax2.set_ylabel("Running Count")
ax2.set_title("Running Detection Count")
ax2.grid(axis="y", alpha=0.3)
stats = ax2.text(0.5, 0.92, "", transform=ax2.transAxes, ha="center", va="top", fontsize=11)

def init():
    """Clear both panels before the first frame."""
    line.set_data([], [])
    dots.set_offsets(np.empty((0, 2)))
    status.set_text("")
    stats.set_text("")
    return line, dots, status, stats, *bars

def update(frame):
    """Reveal one more sample and refresh everything."""
    n = frame + 1                                              # samples revealed so far
    x = np.arange(n)
    y = demo_scores[:n]
    line.set_data(x, y)                                        # update the connecting line
    dots.set_offsets(np.column_stack([x, y]))                  # update dot positions
    dots.set_facecolors(point_colors[:n])                      # colour by true class

    seen_p, seen_t = demo_preds[:n], demo_true[:n]             # predictions and truth so far
    bars[0].set_height(int(np.sum(seen_p == 0)))               # flagged-normal bar
    bars[1].set_height(int(np.sum(seen_p == 1)))               # flagged-anomaly bar

    caught = int(np.sum((seen_p == 1) & (seen_t == 1)))        # attacks correctly flagged
    missed = int(np.sum((seen_p == 0) & (seen_t == 1)))        # attacks that slipped through
    false_alarms = int(np.sum((seen_p == 1) & (seen_t == 0)))  # benign traffic wrongly flagged
    stats.set_text(f"Running accuracy: {np.mean(seen_p == seen_t) * 100:.1f}%   |   "
                   f"Attacks caught: {caught}   |   Missed: {missed}   |   False alarms: {false_alarms}")

    correct = demo_preds[frame] == demo_true[frame]            # was the current sample classified correctly?
    action = "flagged as ANOMALY" if demo_preds[frame] == 1 else "passed as NORMAL"
    status.set_text(f"Sample {n}/{N}: {demo_names[frame]} -> {action} [{'CORRECT' if correct else 'WRONG'}]")
    status.set_color("seagreen" if correct else "firebrick")
    return line, dots, status, stats, *bars

anim = animation.FuncAnimation(fig, update, frames=N, init_func=init,
                               interval=60, blit=False, repeat=False)   # blit=False: we update bars and text
plt.tight_layout()
plt.close(fig)                  # stop Colab showing an extra static copy under the animation
HTML(anim.to_jshtml())          # interactive player with play/pause and a speed slider